In [1]:
import pandas as pd
import numpy as np

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv('../Phishing_Email.csv')
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


# **Data Preprocessing**

In [3]:
df.isnull().sum()

Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64

In [4]:
original_df = df
df = df.dropna(subset=['Email Text'])

print(f"Original rows: {len(original_df)}")
print(f"Cleaned rows: {len(df)}")
print(len(original_df)-len(df))

df.isnull().sum()

Original rows: 18650
Cleaned rows: 18634
16


Unnamed: 0    0
Email Text    0
Email Type    0
dtype: int64

In [5]:

print(df['Email Type'].value_counts())

Email Type
Safe Email        11322
Phishing Email     7312
Name: count, dtype: int64


# **Using XG Boost**

In [6]:
lable_mapping = {
    'Phishing Email':1,
    'Safe Email':0
}

df['Email Type'] = df['Email Type'].map(lable_mapping)
y = df['Email Type']
y

0        0
1        0
2        0
3        1
4        1
        ..
18645    1
18646    0
18647    0
18648    0
18649    1
Name: Email Type, Length: 18634, dtype: int64

In [7]:
#df["Combined_text"] = df["sender_email"]+' '+df["subject"]+' '+df['body']
vectorizer = TfidfVectorizer(stop_words='english',max_features=5000)
x = vectorizer.fit_transform(df['Email Text'])
x
'''
if_url = df["contains_url"].astype(int)
url = if_url.values.reshape(-1,1)

x = hstack([text,url])

#x = text
'''

'\nif_url = df["contains_url"].astype(int)\nurl = if_url.values.reshape(-1,1)\n\nx = hstack([text,url])\n\n#x = text\n'

In [8]:
x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size=0.3,random_state=42)
x_verif,x_test,y_verif,y_test = train_test_split(x_temp,y_temp,test_size=0.5)

In [10]:
best_verif_score = 0
best_model = None
depth_setting = 0

depth_to_test = [3, 6, 9, 12]

for curr_d in depth_to_test:
    model = XGBClassifier(
        n_estimators=100,
        max_depth=curr_d,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    )
    model.fit(x_train,y_train)
    
    y_verif_pred = model.predict(x_verif)
    curr_score = accuracy_score(y_verif,y_verif_pred)
    
    print("*"*30)
    print(f"Depth Setting: {curr_d} accuracy score: {curr_score*100:.2f}")
    
    
    if curr_score > best_verif_score:
        best_verif_score = curr_score
        best_model = model
        best_setting = curr_d
        
print(f"\n\nBest depth setting: {best_setting} best verification score: {best_verif_score}")

******************************
Depth Setting: 3 accuracy score: 92.63
******************************
Depth Setting: 6 accuracy score: 94.35
******************************
Depth Setting: 9 accuracy score: 94.88
******************************
Depth Setting: 12 accuracy score: 95.35


Best depth setting: 12 best verification score: 0.9534883720930233


In [11]:
# 1. Check your dataset balance
print("Dataset Breakdown:")
print(df['Email Type'].value_counts(normalize=True) * 100)

# 2. Look at a detailed report for one of your models
from sklearn.metrics import classification_report
print("\nDetailed Report:")
print(classification_report(y_verif, best_model.predict(x_verif)))

Dataset Breakdown:
Email Type
0    60.759901
1    39.240099
Name: proportion, dtype: float64

Detailed Report:
              precision    recall  f1-score   support

           0       0.99      0.94      0.96      1675
           1       0.91      0.98      0.94      1120

    accuracy                           0.95      2795
   macro avg       0.95      0.96      0.95      2795
weighted avg       0.96      0.95      0.95      2795



In [12]:
y_test_pred = best_model.predict(x_test)
print("=== FINAL TEST RESULTS===")
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_test_pred))

=== FINAL TEST RESULTS===
Test Accuracy: 95.60%

              precision    recall  f1-score   support

           0       0.99      0.94      0.96      1676
           1       0.92      0.98      0.95      1120

    accuracy                           0.96      2796
   macro avg       0.95      0.96      0.95      2796
weighted avg       0.96      0.96      0.96      2796



In [13]:

probabilities = model.predict_proba(x_test)[:, 1]


custom_threshold = 0.80


adjusted_predictions = (probabilities >= custom_threshold).astype(int)


print(classification_report(y_test, adjusted_predictions))

              precision    recall  f1-score   support

           0       0.88      1.00      0.93      1676
           1       0.99      0.80      0.88      1120

    accuracy                           0.92      2796
   macro avg       0.94      0.90      0.91      2796
weighted avg       0.92      0.92      0.91      2796



# **Input**

In [14]:
def check_email(raw_email,threshold=0.80):
    email = vectorizer.transform([raw_email])
    
    prob = model.predict_proba(email)[0,1]
    
    if prob >= threshold:
        return f'Phishing Email (Confidance: {prob:.2f})'
    else:
        return f'Regular Email (risk: {prob:.2f})'


In [15]:
e_mail = input("Enter your email: ")

print(check_email(e_mail))

Regular Email (risk: 0.61)


In [16]:
# Test the already-trained model on phishing_dataset.csv

test_df = pd.read_csv("../phishing_dataset.csv")

# Use the email body because the model was trained on "Email Text"
test_text = test_df["body"].fillna("").astype(str)

# Transform test data using the existing fitted vectorizer
X_external_test = vectorizer.transform(test_text)

# True labels
y_external_test = test_df["is_phishing"].astype(int)

# Predict using the already-trained best model
external_predictions = best_model.predict(X_external_test)

# Display results
print("=== EXTERNAL TEST RESULTS ===")
print(
    f"Accuracy: "
    f"{accuracy_score(y_external_test, external_predictions) * 100:.2f}%\n"
)

print(classification_report(
    y_external_test,
    external_predictions,
    target_names=["Safe Email", "Phishing Email"]
))

=== EXTERNAL TEST RESULTS ===
Accuracy: 52.02%

                precision    recall  f1-score   support

    Safe Email       0.66      0.07      0.13      4967
Phishing Email       0.51      0.96      0.67      5033

      accuracy                           0.52     10000
     macro avg       0.59      0.52      0.40     10000
  weighted avg       0.59      0.52      0.40     10000



In [17]:
import joblib

joblib.dump(vectorizer, 'tfidf_Vectorizer.joblib')
joblib.dump(best_model, 'XGBoost_Phishing_Model.joblib')

print("Model and Vectorizer successfully saved!")

Model and Vectorizer successfully saved!
